# Evaluate related-topic closeness

For **A → B**, rank all other topics by cosine similarity to A and check the rank of expected topic B. Exclude only the query topic. Also evaluate **B → A**: similarity is symmetric, but neighbor ranks need not be. Ties follow embedding-file row order. Validation errors are reported and included as non-hits in the summary.

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("my_data/sgns/topic_embeddings_normalized.csv")
PAIR_PATH = Path("evaluation/topic_related_candidates.csv")
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
DIRECTIONS = {"a_to_b": ("a", "b"), "b_to_a": ("b", "a")}

topics = pd.read_csv(DATA_PATH, keep_default_na=False, dtype={"Topic_Id": str})
embedding_columns = sorted(c for c in topics.columns if c.startswith("embedding_"))
vectors = topics[embedding_columns].to_numpy(dtype=np.float64)
if not embedding_columns or not np.isfinite(vectors).all():
    raise ValueError("Embeddings must contain finite numeric coordinates.")
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
if np.any(norms == 0) or topics["Topic_Id"].duplicated().any():
    raise ValueError("Topic IDs must be unique and vectors must have nonzero length.")
vectors = vectors / norms
paths = topics[TOPIC_COLUMNS].apply(lambda row: "/".join(str(v) for v in row if v), axis=1)
id_to_index = {topic_id: i for i, topic_id in enumerate(topics["Topic_Id"])}
pairs = pd.read_csv(PAIR_PATH, keep_default_na=False, dtype=str)
if pairs["pair_id"].eq("").any() or pairs["pair_id"].duplicated().any():
    raise ValueError("Pair IDs must be nonempty and unique.")
print(f"Loaded {len(topics):,} topics × {len(embedding_columns)} dimensions.")
print(f"Loaded {len(pairs):,} pairs from {PAIR_PATH}.")
print(pairs.groupby("relation").size().rename("Pairs").to_string())

Loaded 1,000 topics × 150 dimensions.
Loaded 645 pairs from evaluation/topic_related_candidates.csv.
relation
close_theme            619
taxonomy_relocation     26


In [10]:
def evaluate_pair(pair, top_k=5, verbose=False, direction="a_to_b"):
    """Accept a pair row or ID; return metrics and neighbors for one direction.

    Set verbose=True for manual inspection. Only the query topic is excluded.
    """
    if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")
    if direction not in DIRECTIONS:
        raise ValueError(f"direction must be one of {tuple(DIRECTIONS)}.")
    if isinstance(pair, str):
        selected = pairs.loc[pairs["pair_id"] == pair]
        if len(selected) != 1:
            raise ValueError(f"Expected exactly one pair with ID {pair}; found {len(selected)}.")
        pair = selected.iloc[0]
    indices = {}
    for role in "ab":
        topic_id = pair[f"topic_id_{role}"]
        if topic_id not in id_to_index:
            raise ValueError(f"Topic {role.upper()} (ID {topic_id}) is missing from the embeddings.")
        index = id_to_index[topic_id]
        if paths.iloc[index] != pair[f"topic_path_{role}"]:
            raise ValueError(f"Path mismatch for {role.upper()}: {paths.iloc[index]} != "
                             f"{pair[f'topic_path_{role}']}")
        indices[role] = index
    source_role, target_role = DIRECTIONS[direction]
    source, target = indices[source_role], indices[target_role]
    if source == target:
        raise ValueError("A related-topic pair must contain two distinct topics.")
    scores = vectors @ vectors[source]
    candidates = np.flatnonzero(np.arange(len(topics)) != source)
    ranking = candidates[np.argsort(-scores[candidates], kind="stable")]
    expected_rank = int(np.flatnonzero(ranking == target)[0]) + 1
    top_indices = ranking[:top_k]
    metrics = {
        "pair_id": pair["pair_id"], "relation": pair["relation"], "direction": direction,
        "query_topic_id": pair[f"topic_id_{source_role}"], "query_path": paths.iloc[source],
        "expected_topic_id": pair[f"topic_id_{target_role}"], "expected_path": paths.iloc[target],
        "expected_rank": expected_rank, "cosine_similarity": float(scores[target]),
        "eligible_topics": len(candidates),
        **{f"top{k}": expected_rank <= k for k in (5, 3, 1)}, "error": "",
    }
    neighbors = pd.DataFrame({
        "Rank": np.arange(1, len(top_indices) + 1),
        "Topic_Id": topics.iloc[top_indices]["Topic_Id"].to_numpy(),
        "Topic_path": paths.iloc[top_indices].to_numpy(),
        "Cosine_similarity": scores[top_indices], "Expected_answer": top_indices == target,
    })
    if verbose:
        print(f"{pair['pair_id']} | Relation: {pair['relation']} | Direction: {direction}")
        print(f"Comments: {pair['comments'] or '(none)'}")
        print(f"Query: {metrics['query_topic_id']} | {metrics['query_path']}")
        print(f"Expected: {metrics['expected_topic_id']} | {metrics['expected_path']}")
        print(neighbors.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
        print(f"Expected-topic rank: {expected_rank} / {len(candidates)}")
        print(f"Pair cosine similarity: {metrics['cosine_similarity']:.6f}")
        print(" | ".join(f"Top {k}: {metrics[f'top{k}']}" for k in (5, 3, 1)))
    return metrics, neighbors


def evaluate_all_pairs(pair_rows):
    """Evaluate every pair in both directions, retaining validation failures."""
    records = []
    for _, pair in pair_rows.iterrows():
        for direction, (source_role, target_role) in DIRECTIONS.items():
            try:
                metrics, _ = evaluate_pair(pair, direction=direction)
            except ValueError as exc:
                metrics = {
                    "pair_id": pair["pair_id"], "relation": pair["relation"], "direction": direction,
                    "query_topic_id": pair[f"topic_id_{source_role}"],
                    "query_path": pair[f"topic_path_{source_role}"],
                    "expected_topic_id": pair[f"topic_id_{target_role}"],
                    "expected_path": pair[f"topic_path_{target_role}"],
                    "expected_rank": None,
                    **{f"top{k}": False for k in (5, 3, 1)}, "error": str(exc),
                }
            records.append(metrics)
    if not records:
        raise ValueError("No pairs to evaluate.")
    results = pd.DataFrame(records)
    results["expected_rank"] = results["expected_rank"].astype("Int64")
    return results


def summarize_pairs(results):
    """Summarize each relation and all pairs, separately for each direction."""
    if results.empty:
        raise ValueError("No results to summarize.")
    groups = list(results.groupby("relation", sort=True)) + [("ALL", results)]
    records = []
    for relation, group in groups:
        for direction, subset in group.groupby("direction", sort=True):
            for k in (5, 3, 1):
                matches = int(subset[f"top{k}"].sum())
                records.append({
                    "Relation": relation, "Direction": direction, "Cutoff": f"Top {k}",
                    "Matches": matches, "Total pairs": len(subset),
                    "Hit rate (%)": 100 * matches / len(subset),
                })
    return pd.DataFrame(records)

## Manually check a pair

Run setup and the function definitions first. Change the ID, neighbor count, or direction below. `evaluate_pair("RP001")` is silent by default and returns `(metrics, neighbors)`; use `verbose=True` to print details. A CSV row can also be passed directly.

In [11]:
PAIR_ID = "RP001"
TOP_K = 5
DIRECTION = "a_to_b"  # Or "b_to_a"

single_metrics, single_neighbors = evaluate_pair(
    PAIR_ID, top_k=TOP_K, verbose=True, direction=DIRECTION
)

RP001 | Relation: taxonomy_relocation | Direction: a_to_b
Comments: automobil was replaced by mobilitaet in 2022
Query: 119 | lifestyle/automobil/autos
Expected: 435 | lifestyle/mobilitaet/autos
 Rank Topic_Id                           Topic_path  Cosine_similarity  Expected_answer
    1      774           lifestyle/automobil/salons           0.798822            False
    2      542       lifestyle/automobil/rondomobil           0.762553            False
    3      184         lifestyle/automobil/eigenbau           0.736625            False
    4      676     lifestyle/automobil/mercedesbenz           0.732552            False
    5      281 lifestyle/automobil/umweltundtechnik           0.731559            False
Expected-topic rank: 16 / 999
Pair cosine similarity: 0.591368
Top 5: False | Top 3: False | Top 1: False


## Evaluate all pairs and summarize

`all_results` has two rows per pair, one for each direction, including expected ranks, cosine similarities, hit flags, and errors. Inspect top-5 misses with `all_results.loc[~all_results["top5"]]`, or filter by `relation` and `direction`. Batch evaluation is independent of the standalone settings above.

In [12]:
all_results = evaluate_all_pairs(pairs)
errors = all_results.loc[all_results["error"].ne(""), ["pair_id", "direction", "error"]]
print(f"Evaluated {len(pairs)} pairs in both directions ({len(all_results)} checks).")
print(f"Successful checks: {len(all_results) - len(errors)}; errors: {len(errors)}.")
if not errors.empty:
    print("\nValidation errors (included as non-hits):")
    print(errors.to_string(index=False))

summary = summarize_pairs(all_results)
print("\nExpected related topics found in the top results:")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

Evaluated 645 pairs in both directions (1290 checks).
Successful checks: 1290; errors: 0.

Expected related topics found in the top results:
           Relation Direction Cutoff  Matches  Total pairs  Hit rate (%)
        close_theme    a_to_b  Top 5      242          619         39.10
        close_theme    a_to_b  Top 3      177          619         28.59
        close_theme    a_to_b  Top 1       77          619         12.44
        close_theme    b_to_a  Top 5      289          619         46.69
        close_theme    b_to_a  Top 3      230          619         37.16
        close_theme    b_to_a  Top 1      118          619         19.06
taxonomy_relocation    a_to_b  Top 5       11           26         42.31
taxonomy_relocation    a_to_b  Top 3       11           26         42.31
taxonomy_relocation    a_to_b  Top 1        5           26         19.23
taxonomy_relocation    b_to_a  Top 5       13           26         50.00
taxonomy_relocation    b_to_a  Top 3       11           

## Main metrics

For each relation, report `(a_to_b top-5 hit rate + b_to_a top-5 hit rate) / 2`, as a percentage.

In [13]:
# Main metrics: equally average the two directional top-5 hit rates.
# Validation errors remain non-hits, as in the full evaluation summary.
relations = ["close_theme", "taxonomy_relocation"]
directional_top5 = (
    all_results.groupby(["relation", "direction"])["top5"]
    .mean()
    .unstack("direction")
    .reindex(index=relations, columns=["a_to_b", "b_to_a"])
)
if directional_top5.isna().any().any():
    raise ValueError("Both directions must have results for each requested relation.")

main_metrics = (
    directional_top5.mean(axis=1).mul(100)
    .rename("Average top-5 hit rate (%)")
    .rename_axis("Relation")
    .reset_index()
)
main_metrics["Relation"] = main_metrics["Relation"].map({
    "close_theme": "Close themes",
    "taxonomy_relocation": "Taxonomy relocations",
})
print(main_metrics.to_string(index=False, float_format=lambda value: f"{value:.2f}"))


            Relation  Average top-5 hit rate (%)
        Close themes                       42.89
Taxonomy relocations                       46.15
